In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import PowerTransformer  # Importa PowerTransformer
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score, hinge_loss
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV



# Caricare i dati dal CSV
file_path = 'Data/features_3_sec.csv'
data = pd.read_csv(file_path)

# Stampa i nomi delle colonne per verificare
#print("Colonne nel DataFrame:", data.columns)

# Separare le caratteristiche e le etichette
X = data.drop(columns=['filename', 'label'])  # Rimuovi filename e label
y = data['label']  # Etichette

# Suddividere il dataset in set di addestramento e test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardizzare le caratteristiche
robust = RobustScaler()
standard = StandardScaler()
minmax = MinMaxScaler()
power = PowerTransformer()
scalers=[robust, standard, minmax, power]




senza nulla

In [18]:
# Definizione del modello di Logistic Regression
model = LogisticRegression(max_iter=100000)
model.fit(X_train, y_train)

    # Valutazione del modello
y_pred = model.predict(X_test)

    # Calcolare metriche
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')  # Macro per bilanciare le classi
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')
f2 = fbeta_score(y_test, y_pred, beta=2, average='macro')  # F2-score con beta=2

    # Stampare i risultati
print(f'Accuratezza: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-score: {f1:.4f}')
print(f'F2-score: {f2:.4f}')
print('------------------------------------------------------------')

Accuratezza: 0.4515
Precision: 0.4310
Recall: 0.4525
F1-score: 0.4266
F2-score: 0.4380
------------------------------------------------------------


c:\Users\romeo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of f AND g EVALUATIONS EXCEEDS LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


con standardizzazione

In [ ]:
for scaler in scalers:

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Definizione del modello di Logistic Regression
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    # Valutazione del modello
    y_pred = model.predict(X_test)

    # Calcolare metriche
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')  # Macro per bilanciare le classi
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    f2 = fbeta_score(y_test, y_pred, beta=2, average='macro')  # F2-score con beta=2

    # Stampare i risultati
    print(f'Scaler: {scaler.__class__.__name__}')
    print(f'Accuratezza: {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')
    print(f'F2-score: {f2:.4f}')
    print('------------------------------------------------------------')

Scaler: RobustScaler
Accuratezza: 0.7347
Precision: 0.7302
Recall: 0.7356
F1-score: 0.7314
F2-score: 0.7336
------------------------------------------------------------
Scaler: StandardScaler
Accuratezza: 0.7362
Precision: 0.7318
Recall: 0.7371
F1-score: 0.7330
F2-score: 0.7351
------------------------------------------------------------
Scaler: MinMaxScaler
Accuratezza: 0.7012
Precision: 0.6956
Recall: 0.7017
F1-score: 0.6950
F2-score: 0.6982
------------------------------------------------------------
Scaler: PowerTransformer
Accuratezza: 0.7467
Precision: 0.7423
Recall: 0.7473
F1-score: 0.7436
F2-score: 0.7456
------------------------------------------------------------


grid search

In [5]:

 #Definizione del pipeline con standardizzazione e Logistic Regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Standardizzazione delle features
    ('classifier', LogisticRegression(max_iter=1000))  # Classificatore Logistic Regression
])

# Definizione degli iperparametri da testare con GridSearch, con combinazioni compatibili
param_grid = [
    {'classifier__C': [0.01, 0.1, 1], 'classifier__penalty': ['l2'], 'classifier__solver': ['lbfgs']},
    {'classifier__C': [0.01, 0.1, 1], 'classifier__penalty': ['l1', 'l2'], 'classifier__solver': ['liblinear']},
    {'classifier__C': [0.01, 0.1, 1], 'classifier__penalty': ['l1', 'l2', 'elasticnet'], 
     'classifier__solver': ['saga'], 'classifier__l1_ratio': [0.5]}  # l1_ratio solo per elasticnet con saga
]

# Inizializzazione della GridSearchCV con validazione incrociata a 5 fold
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)

# Esecuzione della ricerca
grid_search.fit(X_train, y_train)

# Migliori parametri trovati
print("Migliori iperparametri trovati:", grid_search.best_params_)
print("Migliore accuratezza ottenuta:", grid_search.best_score_)

# Valutazione del modello sul set di test
best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print("Accuratezza sul set di test:", test_accuracy)

Fitting 5 folds for each of 18 candidates, totalling 90 fits


KeyboardInterrupt: 

con grid e standardizzazione

In [ ]:
for scaler in scalers:

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Definizione del modello di Logistic Regression
    model = LogisticRegression(multi_class='auto', solver='lbfgs', max_iter=1000)
    model.fit(X_train, y_train)

    # Valutazione del modello
    y_pred = model.predict(X_test)

    # Calcolare metriche
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')  # Macro per bilanciare le classi
    recall = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    f2 = fbeta_score(y_test, y_pred, beta=2, average='macro')  # F2-score con beta=2

    # Stampare i risultati
    print(f'Scaler: {scaler.__class__.__name__}')
    print(f'Accuratezza: {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')
    print(f'F2-score: {f2:.4f}')
    print('------------------------------------------------------------')

c:\Users\romeo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Scaler: RobustScaler
Accuratezza: 0.7357
Precision: 0.7314
Recall: 0.7365
F1-score: 0.7326
F2-score: 0.7346
------------------------------------------------------------


c:\Users\romeo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Scaler: StandardScaler
Accuratezza: 0.7362
Precision: 0.7317
Recall: 0.7371
F1-score: 0.7330
F2-score: 0.7351
------------------------------------------------------------


c:\Users\romeo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Scaler: MinMaxScaler
Accuratezza: 0.7002
Precision: 0.6946
Recall: 0.7008
F1-score: 0.6938
F2-score: 0.6971
------------------------------------------------------------


c:\Users\romeo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Scaler: PowerTransformer
Accuratezza: 0.7477
Precision: 0.7431
Recall: 0.7483
F1-score: 0.7445
F2-score: 0.7465
------------------------------------------------------------
